# Introdução ao deep learning

**Objetivo:** treinar uma pequena rede densa em PyTorch para classificar **dígitos manuscritos** (imagens), comparar com uma regressão logística e discutir, com números, o que a profundidade acrescentou.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)
import torch

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digitos = load_digits()
X = StandardScaler().fit_transform(digitos.data)   # 64 pixels (8x8)
y = digitos.target                                 # 0 a 9
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEMENTE, stratify=y)
print("treino:", X_tr.shape, "| 10 classes de digitos")

## 1. Uma linha de base linear

Antes da rede, uma regressão logística — para sabermos o que a profundidade precisa superar.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

base = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
print("acuracia da regressao logistica:", round(base.score(X_te, y_te), 3))

## 2. A rede densa em PyTorch

`64 → 64 → 32 → 10`, com ReLU nas ocultas. Para 10 classes usamos `CrossEntropyLoss` (que já embute o softmax). Laço de treino explícito, em mini-lotes.

In [ ]:
ent_tr = torch.tensor(X_tr, dtype=torch.float32)
alvo_tr = torch.tensor(y_tr, dtype=torch.long)
ent_te = torch.tensor(X_te, dtype=torch.float32)

torch.manual_seed(SEMENTE)
rede = torch.nn.Sequential(
    torch.nn.Linear(64, 64), torch.nn.ReLU(),
    torch.nn.Linear(64, 32), torch.nn.ReLU(),
    torch.nn.Linear(32, 10))
custo_fn = torch.nn.CrossEntropyLoss()
oti = torch.optim.Adam(rede.parameters(), lr=0.01)

perdas = []
for epoca in range(80):
    ordem = torch.randperm(len(ent_tr))
    for i in range(0, len(ent_tr), 64):
        idx = ordem[i:i+64]
        perda = custo_fn(rede(ent_tr[idx]), alvo_tr[idx])
        oti.zero_grad(); perda.backward(); oti.step()
    perdas.append(perda.item())

with torch.no_grad():
    previsto = rede(ent_te).argmax(dim=1).numpy()
print("acuracia da rede densa:", round(accuracy_score(y_te, previsto), 3))

In [ ]:
figura = go.Figure(go.Scatter(y=perdas, mode="lines", line=dict(color=VERDE)))
figura.update_layout(title="Perda do treino da rede densa (dígitos)",
                     xaxis_title="epoca", yaxis_title="CrossEntropy", height=320,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. O que a profundidade acrescentou?

O resultado é honesto e instrutivo: a rede densa **empata** com a regressão logística — aqui, fica até um fio atrás. Nos dígitos 8×8, já quase linearmente separáveis, a profundidade **não** traz vantagem, exatamente o ponto do texto. O deep learning **decola** mesmo em imagens grandes e cruas (com CNNs) e em texto; numa base pequena e simples, um bom modelo clássico iguala ou supera a rede, com muito menos esforço.

## Exercício

A rede **empatou** (ou perdeu por pouco) para a regressão logística. Em que cenário o deep learning teria vantagem **clara**?

<details><summary>Ver resposta</summary>

Quando os dados são **imagens grandes e cruas** (não 8×8, mas centenas de milhares de pixels), **texto** ou **áudio**, e há **muitos** exemplos. Aí a capacidade do deep learning de aprender **representações hierárquicas** (bordas → partes → objetos), tipicamente com **CNNs** ou **Transformers**, supera de longe qualquer modelo linear sobre pixels crus. Nos dígitos 8×8, quase separáveis, sobra pouco espaço para essa vantagem aparecer.

</details>